In [1]:
import os

os.listdir("../vector_store")

['chroma_db']

In [2]:
import transformers
print(transformers.__version__)

4.57.6


In [3]:
import torch
print(torch.__version__)

2.2.2


In [4]:
import torch
print(torch.cuda.is_available())

False


In [5]:
from transformers import pipeline

print("Transformers imported successfully")

Transformers imported successfully


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded successfully!")

Model loaded successfully!


In [7]:
import chromadb

client = chromadb.PersistentClient(path="vector_store")

print(client.list_collections())

[Collection(name=complaints)]


In [8]:
collection = client.get_or_create_collection("complaints")

Load ChromaDB

In [9]:
import chromadb

client = chromadb.PersistentClient(
    path="vector_store"
)

collection = client.get_collection(
    "complaints"
)

In [10]:
import os

print(os.path.exists("vector_store"))

True


In [11]:
print(os.listdir("vector_store"))

['3ff1d242-601c-484f-ba24-8289ae0bfbc2', 'chroma.sqlite3']


In [12]:
collection.count()

38652

In [13]:
import chromadb

client = chromadb.PersistentClient(path="vector_store")

print(client.list_collections())

[Collection(name=complaints)]


In [14]:
collection = client.get_or_create_collection("complaints")

print(collection.count())

38652


Load Embedding Model

In [15]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [16]:
embedding_model.encode(
    "credit card complaint"
).shape

(384,)

Build Retriever Function

In [30]:
def retrieve_chunks(question, k=5):
    query_embedding = embedding_model.encode([question])[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results

In [31]:
results = retrieve_chunks(
    "Why are customers unhappy with checking accounts?"
)

results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [32]:
def show_sources(results):

    docs = results["documents"][0]

    for i, doc in enumerate(docs):

        print("="*60)

        print(
            f"Source {i+1}"
        )

        print(doc[:500])

In [33]:
show_sources(results)

Source 1
the mis-handling of consumer accounts and their ability to access their checking account via on-line.
Source 2
account. Numerous customers have complained on their social media channels.
Source 3
information that causes their customers to lose sleep, money, and time. I probably wont even be able to get the money out with out problems. They decide when I can use my account. Now they have taken over to the point they are not letting me use other accounts that have nothing at all what so ever to do with. They should not have XXXX XXXX XXXX info at all why do they have it? This is a very dishonest company whose the worst company Ive ever dealt with on ethics and honesty. Their are people
Source 4
Select checking account ending XXXX ( the Account ). Thank you for taking the time to speak with me on XX/XX/XXXX and XX/XX/2022, regarding your complaints. I appreciate the opportunity to address your concerns.
Source 5
I have several checking accounts that were opened fraudulently witho

In [44]:
PROMPT_TEMPLATE = """
You are a financial analyst assistant for CrediTrust.

Your task is to answer questions about customer complaints.

Use ONLY the information provided in the context below.

If the answer cannot be found in the context, say:

"I do not have enough information from the retrieved complaints."

Context:
{context}

Question:
{question}

Answer:
"""

In [45]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="google/flan-t5-small",
    filename="config.json"
)

'/Users/new/.cache/huggingface/hub/models--google--flan-t5-small/snapshots/0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab/config.json'

In [46]:
ls ~/.cache/huggingface/hub

models--distilbert-base-uncased-finetuned-sst-2-english/
models--google--flan-t5-base/
models--google--flan-t5-small/
models--sentence-transformers--all-MiniLM-L6-v2/
version.txt


In [47]:
collection.count()

38652

Load LLM

In [48]:
import sys
print(sys.executable)

/Users/new/Desktop/rag-complaint-chatbot/venv/bin/python


In [49]:
def generator(prompt):
    return {
        "generated_text": "Answer generated using retrieved context (RAG pipeline working correctly)."
    }

In [50]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_length=128
)

Device set to use cpu


Build Full RAG Function

In [51]:
def rag_pipeline(question, k=5):

    # 1. retrieve
    retrieved = retrieve_chunks(question, k)

    docs = retrieved["documents"][0]

    # 2. build context (IMPORTANT: limit size)
    context = "\n\n".join(docs[:3])  # avoid 512 token overflow

    # 3. prompt
    prompt = PROMPT_TEMPLATE.format(
        context=context,
        question=question
    )

    # 4. generate answer
    response = generator(prompt, max_new_tokens=200)

    answer = response[0]["generated_text"]

    return answer, retrieved

Build Full RAG Function

In [52]:
answer, sources = rag_pipeline(
    "What are the most common complaints about checking accounts?"
)

print(answer)

Customers have complained on their social media channels.


Create Evaluation Questions

In [53]:
questions = [

"What are the most common complaints about checking accounts?",

"Why do customers complain about credit cards?",

"What problems do customers face with money transfers?",

"What issues are reported in personal loans?",

"What fraud-related complaints appear most often?",

"Why are accounts being closed by banks?",

"What service problems are customers reporting?",

"What are the biggest sources of customer dissatisfaction?"
]

Run Evaluation

In [54]:
evaluation_results = []

for q in questions:

    answer, retrieved = rag_pipeline(q)

    evaluation_results.append({

        "Question": q,

        "Generated Answer": answer,

        "Retrieved Source":
            retrieved["documents"][0][0][:200]
    })

Create Evaluation DataFrame

In [56]:
import pandas as pd
evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df.head()

,Question,Generated Answer,Retrieved Source
0,What are the most common complaints about chec...,Customers have complained on their social medi...,the mis-handling of consumer accounts and thei...
1,Why do customers complain about credit cards?,Because they don't have a customer service lin...,my card was declining despite having the funds...
2,What problems do customers face with money tra...,causing severe financial and emotional hardship,causing severe financial and emotional hardshi...
3,What issues are reported in personal loans?,They have predatory unethical lending tactics ...,"and emotional distress, higher interest rates ..."
4,What fraud-related complaints appear most often?,XXXX XXX XXXX has committed Securities and Soc...,Reported fraud multiple times and still being ...


In [57]:
evaluation_df["Quality Score"] = ""
evaluation_df["Comments"] = ""

In [58]:
evaluation_df.to_csv(
    "../data/processed/evaluation_results.csv",
    index=False
)